# A continuous ring attractor in the head-direction system, maintained internally during sleep

**Dataset:** [DANDI:000056](https://dandiarchive.org/dandiset/000056) — *Internally organized
mechanisms of the head direction sense* (Peyrache, Lacroix, Petersen & Buzsáki,
*Nature Neuroscience* 2015). Extracellular recordings from anterior thalamus and
post-subiculum in freely moving mice, with an open-field foraging session flanked by
sleep, and REM / non-REM / awake state labels.

## The claim being tested

The head-direction (HD) system is usually described as a *continuous attractor*: a
population of cells whose joint activity is confined to a one-dimensional closed curve
(a ring), with one localised bump of activity whose position on that ring is the
animal's heading. Two things make this more than a redescription of the tuning curves:

1. The ring should be **intrinsic to the network**, not imposed by the sensory input.
   If the ring is generated by recurrent connectivity, it should still be there when
   the animal is asleep and the vestibular and visual heading signals are absent or
   uninformative.
2. The state on the ring should move **continuously**. A collection of independently
   modulated direction-selective neurons could produce decoded headings that jump
   arbitrarily from one moment to the next; a state on an attractor cannot leave the
   ring, so it has to travel around it.

This notebook tests both, using only spikes. Every sleep analysis below is computed
without reference to any behavioural measurement, so it cannot be inherited from the
animal's actual head movements.

## What is done

* Identify HD cells from wake tuning curves, with a circular-shift null and a
  split-half stability criterion.
* Show that the pairwise correlation structure of the HD ensemble is a cosine function
  of the difference in preferred direction, in wake **and** in REM and non-REM sleep.
* Embed the binned population activity with Isomap and show that the point cloud is a
  hollow ring in all three states, and a blob for time-shuffled data and for non-HD units.
* Decode an internal heading from the wake tuning curves during sleep, and show that
  two disjoint halves of the ensemble agree with each other (an internal coherence
  measure that never uses behaviour).
* Show that the angular coordinate of the Isomap ring is the same coordinate as the
  decoded heading, and that the internal heading moves smoothly rather than jumping.
* Repeat the whole thing on 4 sessions from 3 mice.

## Setup

Data are streamed from the DANDI S3 bucket with `remfile` and a local disk cache; no
file is downloaded in full. Analysis is done with Pynapple.

In [ ]:
import warnings, os
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import h5py, remfile
from pynwb import NWBHDF5IO
import pynapple as nap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.ndimage import gaussian_filter1d
from sklearn.decomposition import PCA
from sklearn.manifold import Isomap

TWO_PI = 2 * np.pi

## Loading

Head direction is not stored directly in these files. It is recovered from the two
head-mounted LEDs: the vector from the red to the blue LED gives the heading, and
samples where either LED failed to track (coded as -1) are dropped. Brain-state
intervals (`Awake` / `REM` / `Non-REM`) come from the `behavior/states` table. The
wake epoch used for tuning curves is further restricted to periods with continuous
tracking, which selects the open-field foraging part of the session.

The units table in this conversion carries only spike times, with no anatomical label,
so ADn and PoSub cells are pooled and HD cells are identified functionally.

In [ ]:
import h5py, remfile, numpy as np, pandas as pd
from pynwb import NWBHDF5IO
import pynapple as nap

DANDISET = "000056"
VERSION = "0.250624.0430"

# session_id -> asset_id
SESSIONS = {
    "Mouse12-120806": "33f6aa51-f82f-4467-95b0-3f1403668654",
    "Mouse12-120807": "69655198-97af-4e65-9198-1c94d0830d24",
    "Mouse12-120808": "88587645-9aa1-4247-84b2-69bd45a23ce9",
    "Mouse12-120809": "c816972c-f501-4726-9333-a42aa2321d90",
    "Mouse12-120810": "00bf96d4-2c4d-44e9-93ea-a21e7c6e6959",
    "Mouse17-130125": "e62179fc-cd8d-4901-98aa-ccb35086f9f7",
    "Mouse17-130128": "4cc64fe0-7b1e-404c-8b86-fb5659292830",
    "Mouse17-130129": "533ebd09-8af4-4238-8c51-8a679c427c96",
    "Mouse17-130130": "e562f907-b93d-49fa-a1b1-3a3a89645e4f",
    "Mouse17-130131": "d6b3126c-2c99-45b4-a1e6-8f0e5acda578",
    "Mouse17-130201": "7f408e4a-3218-4a17-983b-d7b5cdb9dc82",
    "Mouse17-130202": "c86c7e25-fa65-4356-a699-4b1ab8ab0d64",
    "Mouse17-130203": "74079bfe-223b-45ea-b71b-40d7b2ff6151",
    "Mouse17-130204": "f92f2709-4469-4c1d-a883-75eb66ee898a",
    "Mouse20-130514": "748aa5de-c0de-4aa7-a7ef-2aad2f87a7eb",
    "Mouse20-130515": "f1a2857b-22b7-4e9e-a6af-211c76081391",
    "Mouse20-130516": "714ae657-90a5-44ae-82c5-c1f7e762c2b2",
    "Mouse20-130517": "d3f6aa86-20fb-4330-97dc-8e4900e16a0e",
    "Mouse20-130520": "c0104ace-0807-473a-ad59-a8cc594ac4e9",
    "Mouse24-131213": "ada02790-6eb6-48ee-902d-9ba017303586",
    "Mouse24-131216": "14dafd30-f9fb-4aac-bc8e-e0d46d1b35e6",
    "Mouse24-131217": "c6cfc0f2-dda6-4136-be2c-be4a4de5a50e",
    "Mouse24-131218": "07ef959c-6778-4401-9434-641b88fc75a2",
    "Mouse25-140123": "bdb30f7d-ba69-4d2e-8504-2efab69cd8d7",
    "Mouse25-140124": "43c07029-2ac8-4900-9e51-8c44c3264080",
    "Mouse25-140128": "91242a8e-2402-40b3-926b-8d07aa406748",
    "Mouse25-140129": "8750d43b-c61d-461c-a30c-e3f071a835e6",
    "Mouse25-140130": "9c49d0c6-6e94-405b-b4e9-cb2ac744a36a",
    "Mouse25-140131": "dba64364-e09b-4ad5-b54c-baaa689e975a",
    "Mouse25-140203": "ac07ef7d-3c95-407d-ba01-f872b756b46c",
    "Mouse25-140204": "0889d7ae-7db5-4304-90ca-064cb7671791",
    "Mouse25-140205": "48dae2dc-35f7-4a9e-bcdb-883a7d5e2d32",
    "Mouse25-140206": "2a2e6913-5101-438d-bada-9f7dd46d3507",
    "Mouse28-140310": "656704ea-a4cd-40f4-8158-a6533ebf2eee",
    "Mouse28-140311": "2c0950ed-43c6-4331-adbd-37a6c764758a",
    "Mouse28-140312": "ff614bbb-8194-4372-9c7e-bcc0966bd453",
    "Mouse28-140313": "56d2e2d7-ba41-40a1-b017-b81871f3f0c3",
    "Mouse28-140317": "3c4d6960-bdea-41da-a98f-842be83abf41",
    "Mouse28-140318": "174c0da6-6216-4f12-862b-0a082ce847df",
    "Mouse32-140820": "43be9315-c73f-4e83-adea-20e73ce75466"
}

def asset_url(asset_id):
    return (f"https://api.dandiarchive.org/api/dandisets/{DANDISET}/versions/"
            f"{VERSION}/assets/{asset_id}/download/")

def open_session(asset_id, cache_dir="/tmp/remfile_cache"):
    rf = remfile.File(asset_url(asset_id), disk_cache=remfile.DiskCache(cache_dir))
    h5 = h5py.File(rf, "r")
    io = NWBHDF5IO(file=h5, load_namespaces=True)
    return io.read()

def load_session(asset_id, **kw):
    """Return dict with spikes (TsGroup), hd (Tsd, radians), epochs (dict of IntervalSet)."""
    nwbfile = open_session(asset_id, **kw)
    nwb = nap.NWBFile(nwbfile)
    spikes = nwb["units"]

    blue = nwb["BlueLED"]      # TsdFrame (x, y)
    red = nwb["RedLED"]
    b = blue.values.astype(float)
    r = red.values.astype(float)
    t = blue.index.values
    valid = (b[:, 0] > 0) & (b[:, 1] > 0) & (r[:, 0] > 0) & (r[:, 1] > 0)
    d = b[valid] - r[valid]
    ang = np.arctan2(d[:, 1], d[:, 0]) % (2 * np.pi)
    hd = nap.Tsd(t=t[valid], d=ang)

    states = nwbfile.processing["behavior"]["states"].to_dataframe()
    epochs = {}
    for lab in states["label"].unique():
        s = states[states["label"] == lab]
        epochs[lab] = nap.IntervalSet(start=s["start_time"].values, end=s["stop_time"].values)

    # Wake epoch with valid tracking (the open-field foraging session)
    dt = np.median(np.diff(t))
    gaps = np.diff(hd.index.values)
    brk = np.where(gaps > 5 * dt)[0]
    starts = np.concatenate([[hd.index.values[0]], hd.index.values[brk + 1]])
    ends = np.concatenate([hd.index.values[brk], [hd.index.values[-1]]])
    tracked = nap.IntervalSet(start=starts, end=ends).drop_short_intervals(1.0)
    epochs["wake_tracked"] = tracked.intersect(epochs["Awake"])

    return dict(session_id=nwbfile.session_id, subject=nwbfile.subject.subject_id,
                spikes=spikes, hd=hd, epochs=epochs, nwbfile=nwbfile)

## Identifying head-direction cells

A unit is called an HD cell when its wake firing rate is in a plausible single-unit
range (0.5-50 Hz), its mean vector length exceeds 0.25 *and* the 99th percentile of a
circular-shift null, and its tuning curve is stable across the two halves of the wake
epoch (split-half correlation > 0.5). The null shifts the whole head-direction
sequence relative to the spikes, so it preserves both the occupancy map and each
unit's spike count, and destroys only the alignment between them.

In [ ]:
NBINS = 60
TWO_PI = 2 * np.pi
ANG = (np.arange(NBINS) + 0.5) * TWO_PI / NBINS
EDGES = np.linspace(0, TWO_PI, NBINS + 1)


def tuning(spikes, hd, ep, nb_bins=NBINS):
    """Head-direction tuning curves, (nb_bins, n_units) DataFrame in Hz."""
    return nap.compute_tuning_curves(spikes, hd, bins=nb_bins, range=(0, TWO_PI),
                                     epochs=ep, return_pandas=True)


def mvl_pref(tc_values):
    """Mean vector length and preferred direction from a (nbins, nunits) array."""
    v = np.nan_to_num(np.asarray(tc_values, dtype=float))
    tot = v.sum(0)
    z = (v * np.exp(1j * ANG)[:, None]).sum(0) / np.where(tot > 0, tot, np.nan)
    return np.abs(z), np.angle(z) % TWO_PI


def _spike_hd_index(spikes, hd, ep):
    """For each unit, the index of the nearest HD sample at each of its spikes."""
    hd_ep = hd.restrict(ep)
    ht = hd_ep.index.values
    out = {}
    for u in spikes.keys():
        st = spikes[u].restrict(ep).index.values
        idx = np.searchsorted(ht, st)
        idx = np.clip(idx, 0, len(ht) - 1)
        out[u] = idx
    return hd_ep, out


def screen_hd_cells(spikes, hd, wake, min_rate=0.5, max_rate=50.0,
                    mvl_thresh=0.25, n_shuffle=500, stab_thresh=0.5, seed=0):
    """Per-unit head-direction statistics with an `is_hd` flag.

    A unit counts as an HD cell when (a) its wake firing rate is in a plausible
    single-unit range, (b) its mean vector length exceeds `mvl_thresh` and the
    99th percentile of a circular-shift null, and (c) tuning curves built from
    the two halves of the wake epoch correlate above `stab_thresh`.
    """
    rng = np.random.default_rng(seed)
    tc = tuning(spikes, hd, wake)
    mvl, pref = mvl_pref(tc.values)
    rate = np.array([len(spikes[u].restrict(wake)) / wake.tot_length() for u in spikes.keys()])

    # split-half stability across the two halves of the wake epoch
    t0, t1 = wake.start, wake.end
    k = np.searchsorted(np.cumsum(t1 - t0), wake.tot_length() / 2)
    h1 = nap.IntervalSet(start=t0[:k + 1], end=t1[:k + 1])
    h2 = nap.IntervalSet(start=t0[k + 1:], end=t1[k + 1:])
    tc1 = np.nan_to_num(tuning(spikes, hd, h1).values)
    tc2 = np.nan_to_num(tuning(spikes, hd, h2).values)
    stab = np.array([np.corrcoef(tc1[:, i], tc2[:, i])[0, 1] for i in range(tc1.shape[1])])

    # circular-shift null: shift each unit's spike-to-HD assignment as a block,
    # which preserves both the occupancy map and the unit's spike-count
    hd_ep, sidx = _spike_hd_index(spikes, hd, wake)
    hd_bin = np.clip(np.digitize(hd_ep.values, EDGES) - 1, 0, NBINS - 1)
    dt = np.median(np.diff(hd_ep.index.values))
    occ = np.bincount(hd_bin, minlength=NBINS) * dt
    null = np.zeros((n_shuffle, len(sidx)))
    n_hd = len(hd_bin)
    for j in range(n_shuffle):
        shift = rng.integers(n_hd)
        rolled = np.roll(hd_bin, shift)
        for i, u in enumerate(spikes.keys()):
            cnt = np.bincount(rolled[sidx[u]], minlength=NBINS)
            null[j, i] = mvl_pref((cnt / occ)[:, None])[0][0]
    p99 = np.nanpercentile(null, 99, axis=0)

    df = pd.DataFrame(dict(unit=list(spikes.keys()), rate=rate, mvl=mvl, pref=pref,
                           stability=stab, mvl_null99=p99))
    df["is_hd"] = ((df.rate > min_rate) & (df.rate < max_rate) & (df.mvl > mvl_thresh)
                   & (df.mvl > df.mvl_null99) & (df.stability > stab_thresh))
    return df, tc

## Ring-attractor measurements

Four independent measurements are computed per brain state, all on 100 ms bins of the
HD-cell ensemble smoothed with a 200 ms Gaussian:

* **Cosine correlation structure.** For every pair of HD cells, the correlation of
  their binned firing against the difference of their wake preferred directions. On a
  ring attractor this is a cosine: cells with nearby preferred directions are
  co-active, cells with opposite preferred directions are anti-correlated.
* **Manifold shape.** Isomap embedding of the population vectors, summarised by
  *hollowness* = mean radius / SD of radius, and by how sharply the angular position on
  the ring determines the decoded heading.
* **A single angular coordinate.** The heading decoded from one random half of the
  ensemble is used to predict the activity of the other, disjoint half. Because the
  predictor and the predicted neurons never overlap, the R² cannot come from fitting
  the angle to the spikes it is scored on.
* **Internal coherence.** Two disjoint halves each decode a heading independently
  (Poisson MAP against the wake tuning curves); the mean resultant length of the
  difference between the two decoded angles measures whether the population holds a
  single consistent angular state. This never uses behaviour.

Note on dimensionality: the raw participation ratio of 100 ms binned activity stays
high (around 12 for a 25-cell ensemble) because single-trial spike counts are
noise-dominated, so it is *not* used as evidence for a low-dimensional code. What the
PCA spectrum does show is that the two leading components of the HD ensemble carry
nearly equal variance, as they must if the signal lies in a plane and traverses a ring
in it, whereas for the non-HD control PC1 dominates PC2.

The null throughout is an independent random circular time shift per neuron. It keeps
every single-neuron statistic (rate, autocorrelation, burstiness) and destroys only
the coordination between neurons.

In [ ]:
TWO_PI = 2 * np.pi


# ---------------------------------------------------------------- binning ---
def population_matrix(spikes, ep, bin_size, smooth_sd_bins=1.0):
    """Binned, mildly smoothed spike counts. Returns (times, counts array)."""
    cnt = spikes.count(bin_size, ep)
    x = np.asarray(cnt.values, dtype=float)
    if smooth_sd_bins > 0:
        # smooth inside each contiguous interval so we never blur across gaps
        edges = np.searchsorted(cnt.index.values, ep.end)
        start = 0
        for e in edges:
            if e > start:
                x[start:e] = gaussian_filter1d(x[start:e], smooth_sd_bins, axis=0)
            start = e
    return cnt.index.values, x


def zscore_cols(x):
    m, s = x.mean(0), x.std(0)
    return (x - m) / np.where(s > 0, s, 1.0)


# ------------------------------------------------------------- statistics ---
def circ_diff(a, b):
    """Signed angular difference wrapped to (-pi, pi]."""
    return (a - b + np.pi) % TWO_PI - np.pi


def circ_corr(a, b):
    """Circular-circular correlation coefficient (Jammalamadaka & SenGupta)."""
    a = np.asarray(a); b = np.asarray(b)
    ma = np.angle(np.exp(1j * a).mean())
    mb = np.angle(np.exp(1j * b).mean())
    num = np.sum(np.sin(a - ma) * np.sin(b - mb))
    den = np.sqrt(np.sum(np.sin(a - ma) ** 2) * np.sum(np.sin(b - mb) ** 2))
    return num / den


def pairwise_corr_vs_angle(x, pref):
    """Pearson correlation of every HD-cell pair against their |Δ preferred direction|."""
    c = np.corrcoef(x.T)
    n = c.shape[0]
    iu = np.triu_indices(n, 1)
    d = np.abs(circ_diff(pref[:, None], pref[None, :]))[iu]
    return d, c[iu], c


def cosine_fit(d, r):
    """Least-squares fit r = a + b*cos(d). Returns (a, b, R^2)."""
    X = np.column_stack([np.ones_like(d), np.cos(d)])
    coef, *_ = np.linalg.lstsq(X, r, rcond=None)
    pred = X @ coef
    ss_res = np.sum((r - pred) ** 2)
    ss_tot = np.sum((r - r.mean()) ** 2)
    return coef[0], coef[1], 1 - ss_res / ss_tot


# --------------------------------------------------------------- decoding ---
def bayesian_decode(counts, tc, bin_size):
    """Poisson MAP decoding of heading from binned counts.

    counts : (T, N) spike counts; tc : (B, N) wake tuning curves in Hz.
    Returns (decoded angle (T,), posterior (T, B)).
    """
    tc = np.clip(np.nan_to_num(tc, nan=0.0), 1e-3, None)
    logtc = np.log(tc)
    ll = counts @ logtc.T - bin_size * tc.sum(1)[None, :]
    ll -= ll.max(1, keepdims=True)
    post = np.exp(ll)
    post /= post.sum(1, keepdims=True)
    b = tc.shape[0]
    ang = (np.arange(b) + 0.5) * TWO_PI / b
    z = post @ np.exp(1j * ang)
    return np.angle(z) % TWO_PI, post


def agreement(a, b):
    """Mean resultant length of the angular difference between two circular signals.

    1 means the two signals report the same angle up to a fixed offset, 0 means they
    are unrelated. Preferred here over a circular correlation coefficient, which is
    unstable when the marginal distributions are close to uniform (as they are when
    the bump sweeps the whole ring during sleep).
    """
    return float(np.abs(np.exp(1j * (np.asarray(a) - np.asarray(b))).mean()))


def split_half_decode(counts, tc, bin_size, rng, win=3):
    """Decode heading independently from two disjoint halves of the ensemble.

    Agreement between the two decoders is an *internal* coherence measure: it
    never references the animal's actual head direction, so it is meaningful
    during sleep. Counts are pooled over `win` bins because each half holds only
    about a dozen cells.
    """
    n = counts.shape[1]
    idx = rng.permutation(n)
    a, b = idx[: n // 2], idx[n // 2:]
    c = uniform_filter1d(np.asarray(counts, float), win, axis=0) * win
    da, _ = bayesian_decode(c[:, a], tc[:, a], bin_size * win)
    db, _ = bayesian_decode(c[:, b], tc[:, b], bin_size * win)
    return da, db


def cv_one_d_explained_variance(counts, x, tc, bin_size, rng, n_rep=4, decode_bins=3):
    """Cross-validated test that a *single* angular coordinate describes the ensemble.

    The heading is decoded from one random half of the HD cells and is then used to
    predict the activity of the *other*, disjoint half. Because the predictor and the
    predicted neurons never overlap, a high R^2 cannot come from fitting the angle to
    the same spikes it is being scored on.
    """
    n = counts.shape[1]
    # decode from a slightly wider window than a single bin, to match the 200 ms
    # smoothing of the activity being predicted
    c = uniform_filter1d(counts, decode_bins, axis=0) * decode_bins
    scores = []
    for _ in range(n_rep):
        idx = rng.permutation(n)
        a, b = idx[: n // 2], idx[n // 2:]
        for src, tgt in ((a, b), (b, a)):
            th, _ = bayesian_decode(c[:, src], tc[:, src], bin_size * decode_bins)
            scores.append(np.nanmean(one_d_explained_variance(x[:, tgt], th)))
    return float(np.mean(scores))


# -------------------------------------------------------------- manifolds ---
def manifold_input(counts, smooth_sd_bins=2.0, activity_pct=20):
    """Preprocess binned counts for manifold estimation.

    Square-root transform (variance stabilisation), Gaussian smoothing, removal of the
    lowest-activity bins (during non-REM these are DOWN states in which the population
    is essentially silent and carries no angle), z-scoring per neuron and finally
    normalisation of each population vector to unit length. The last step removes
    variation in overall population gain, which would otherwise appear as a radial
    dimension on top of the ring.
    """
    x = gaussian_filter1d(np.sqrt(np.asarray(counts, dtype=float)), smooth_sd_bins, axis=0)
    act = x.sum(1)
    keep = act > np.percentile(act, activity_pct)
    z = zscore_cols(x[keep])
    return keep, z / np.linalg.norm(z, axis=1, keepdims=True)


def taubin_center(xy):
    """Algebraic circle fit; a better ring centre than the centroid when the points
    are unevenly distributed around the ring."""
    mx, my = xy[:, 0].mean(), xy[:, 1].mean()
    x, y = xy[:, 0] - mx, xy[:, 1] - my
    A = np.column_stack([x, y, np.ones_like(x)])
    c, *_ = np.linalg.lstsq(A, x ** 2 + y ** 2, rcond=None)
    return np.array([c[0] / 2 + mx, c[1] / 2 + my])


def map_concentration(th_x, th_y, nbins=36):
    """How sharply th_y is determined by th_x, for two circular variables.

    th_y is predicted by its circular mean within bins of th_x; the statistic is the
    mean resultant length of the residual. 1 means th_x fixes th_y exactly, 0 means it
    says nothing. Unlike a circular correlation coefficient this is insensitive to
    monotone distortions of either angle, which matters because Isomap does not
    preserve angular spacing.
    """
    b = np.clip((th_x / TWO_PI * nbins).astype(int), 0, nbins - 1)
    fit = np.full_like(th_y, np.nan)
    for k in range(nbins):
        m = b == k
        if m.sum() > 3:
            fit[m] = np.angle(np.exp(1j * th_y[m]).mean())
    resid = th_y - fit
    ok = np.isfinite(resid)
    return float(np.abs(np.exp(1j * resid[ok]).mean()))


def ring_embedding(x, n_neighbors=25, n_components=2, max_points=3000, seed=0):
    """Isomap embedding of population states, subsampled for tractability."""
    rng = np.random.default_rng(seed)
    if x.shape[0] > max_points:
        sel = np.sort(rng.choice(x.shape[0], max_points, replace=False))
    else:
        sel = np.arange(x.shape[0])
    emb = Isomap(n_neighbors=n_neighbors, n_components=n_components).fit_transform(x[sel])
    return sel, emb


def ring_stats(emb):
    """Hollowness of a 2-D point cloud: mean radius / SD of radius.

    An isotropic Gaussian blob gives ~1.9 and a uniform filled disk ~2.8;
    points concentrated on a circle give much larger values. The per-session
    time-shift null is the reference actually used in the figures.
    """
    e = emb - taubin_center(emb)
    r = np.hypot(e[:, 0], e[:, 1])
    theta = np.arctan2(e[:, 1], e[:, 0]) % TWO_PI
    # angular coverage: fraction of 36 angular sectors that are occupied
    occ = np.unique((theta / TWO_PI * 36).astype(int)).size / 36
    return dict(mean_r=r.mean(), cv_r=r.std() / r.mean(),
                hollowness=r.mean() / r.std(), coverage=occ, theta=theta)


def one_d_explained_variance(x, theta):
    """Variance of each neuron's activity explained by a circular tuning to `theta`.

    Regresses activity on [1, cos, sin, cos2, sin2]; the R^2 says how much of the
    population activity is a function of a single angular coordinate.
    """
    X = np.column_stack([np.ones_like(theta), np.cos(theta), np.sin(theta),
                         np.cos(2 * theta), np.sin(2 * theta)])
    coef, *_ = np.linalg.lstsq(X, x, rcond=None)
    pred = X @ coef
    ss_res = ((x - pred) ** 2).sum(0)
    ss_tot = ((x - x.mean(0)) ** 2).sum(0)
    return 1 - ss_res / np.where(ss_tot > 0, ss_tot, np.nan)


def dimensionality_input(counts, smooth_sd_bins=2.0):
    """Square-root transformed, smoothed, z-scored activity for dimensionality measures.

    Raw 100 ms spike counts are dominated by Poisson noise, which is full-rank and
    would swamp any low-dimensional structure; the smoothing is what makes the
    participation ratio informative.
    """
    return zscore_cols(gaussian_filter1d(np.sqrt(np.asarray(counts, dtype=float)),
                                         smooth_sd_bins, axis=0))


def participation_ratio(x):
    ev = PCA().fit(x).explained_variance_
    return ev.sum() ** 2 / (ev ** 2).sum()


def pca_spectrum(x, k=10):
    p = PCA(n_components=min(k, x.shape[1])).fit(x)
    return p.explained_variance_ratio_


# ----------------------------------------------------------------- nulls ----
def shift_shuffle(x, rng, min_shift=50):
    """Circularly shift each neuron's binned activity by an independent random lag.

    Preserves single-neuron statistics, destroys population coordination.
    """
    out = np.empty_like(x)
    t = x.shape[0]
    for i in range(x.shape[1]):
        out[:, i] = np.roll(x[:, i], rng.integers(min_shift, t - min_shift))
    return out

## Per-session pipeline

In [ ]:
warnings.filterwarnings("ignore")


BIN = 0.1          # s
SMOOTH_SD = 2.0    # bins -> 200 ms Gaussian, used for correlations and PCA
MANIFOLD_SD = 2.0  # bins, smoothing used for the Isomap embedding
ACT_PCT = 20       # drop the least active 20% of bins before embedding
STATES = ["wake", "REM", "nREM"]


def nearest_hd(hd, t, tol):
    """Head direction sampled at times `t`, NaN where the nearest sample is > tol away."""
    ht, hv = hd.index.values, hd.values
    i = np.clip(np.searchsorted(ht, t), 1, len(ht) - 1)
    left = np.abs(t - ht[i - 1]) < np.abs(ht[i] - t)
    j = np.where(left, i - 1, i)
    v = hv[j].astype(float)
    v[np.abs(t - ht[j]) > tol] = np.nan
    return v


def analyze(sid, seed=0, n_null=20, verbose=True):
    rng = np.random.default_rng(seed)
    s = load_session(SESSIONS[sid])
    spikes, hd, ep = s["spikes"], s["hd"], s["epochs"]
    wake = ep["wake_tracked"]
    eps = {"wake": wake, "REM": ep["REM"], "nREM": ep["Non-REM"]}

    df, tc = screen_hd_cells(spikes, hd, wake, n_shuffle=200, seed=seed)
    hd_units = df.loc[df.is_hd, "unit"].values
    pref = df.loc[df.is_hd, "pref"].values
    order = np.argsort(pref)
    if verbose:
        print(f"{sid}: {len(spikes)} units, {len(hd_units)} HD cells")
    if len(hd_units) < 8:
        raise RuntimeError(f"{sid}: too few HD cells ({len(hd_units)})")

    hd_spikes = spikes[list(hd_units)]
    tc_hd = np.nan_to_num(tc[hd_units].values)          # (nbins, n_hd), Hz

    # control ensemble: non-HD units with a comparable firing-rate range
    ctrl = df.loc[~df.is_hd & (df.rate > 0.5) & (df.rate < 50), "unit"].values
    ctrl_spikes = spikes[list(ctrl)] if len(ctrl) >= 8 else None

    out = dict(sid=sid, subject=s["subject"], n_units=len(spikes),
               hd_units=hd_units, pref=pref, order=order, tc_hd=tc_hd,
               screen=df.to_dict("list"), bin=BIN,
               ep_len={k: float(v.tot_length()) for k, v in eps.items()})

    for st in STATES:
        e = eps[st]
        t, x = population_matrix(hd_spikes, e, BIN, SMOOTH_SD)
        keep = x.sum(1) > 0                              # drop wholly silent bins
        t, x = t[keep], x[keep]
        z = zscore_cols(x)

        # --- pairwise correlation structure ---
        d, r, cmat = pairwise_corr_vs_angle(x, pref)
        a, b, r2 = cosine_fit(d, r)

        # --- Bayesian decoding against the wake tuning curves ---
        counts = np.asarray(hd_spikes.count(BIN, e).values, dtype=float)[keep]
        dec, post = bayesian_decode(counts, tc_hd, BIN)
        da, db = split_half_decode(counts, tc_hd, BIN, np.random.default_rng(seed))
        coh = agreement(da, db)

        # --- manifold ---
        mkeep, mx = manifold_input(counts, MANIFOLD_SD, ACT_PCT)
        sel, emb = ring_embedding(mx, seed=seed)
        rs = ring_stats(emb)
        theta_emb = rs.pop("theta")
        dec_m = dec[mkeep][sel]
        map_conc = map_concentration(theta_emb, dec_m)
        null_map = np.mean([map_concentration(theta_emb, rng.permutation(dec_m))
                            for _ in range(10)])
        ev1d = cv_one_d_explained_variance(counts, z, tc_hd, BIN,
                                              np.random.default_rng(seed))
        dz = dimensionality_input(counts, SMOOTH_SD)
        spec = pca_spectrum(dz, 10)
        pr = participation_ratio(dz)

        # angular drift speed of the internal bump (rad/s), across contiguous bins
        gap = np.diff(t) > 1.5 * BIN
        step = np.abs(circ_diff(dec[1:], dec[:-1])) / BIN
        step = step[~gap]

        # --- shuffle null: independent circular time shifts per neuron ---
        null_r2, null_coh, null_holl, null_ev = [], [], [], []
        for _ in range(n_null):
            xs = shift_shuffle(x, rng)
            ds, rs_, _ = pairwise_corr_vs_angle(xs, pref)
            null_r2.append(cosine_fit(ds, rs_)[2])
            cs = shift_shuffle(counts, rng)
            na, nb = split_half_decode(cs, tc_hd, BIN, np.random.default_rng(seed))
            null_coh.append(agreement(na, nb))
            null_ev.append(cv_one_d_explained_variance(
                cs, zscore_cols(xs), tc_hd, BIN, np.random.default_rng(seed), n_rep=1))
            _, mxs = manifold_input(cs, MANIFOLD_SD, ACT_PCT)
            _, semb = ring_embedding(mxs, max_points=1500, seed=seed)
            null_holl.append(ring_stats(semb)["hollowness"])

        res = dict(t=t, x=x, z=z, counts=counts, corr_d=d, corr_r=r, cmat=cmat,
                   cos_a=a, cos_b=b, cos_r2=r2,
                   emb=emb, emb_keep=mkeep, emb_sel=sel, theta_emb=theta_emb, ev1d=ev1d,
                   map_conc=map_conc, null_map_conc=null_map,
                   pca_spectrum=spec, participation_ratio=pr,
                   hollowness=rs["hollowness"], cv_r=rs["cv_r"], coverage=rs["coverage"],
                   decoded=dec, decoded_emb=dec_m, split_coh=coh, drift=step,
                   null_cos_r2=np.array(null_r2), null_coh=np.array(null_coh),
                   null_hollowness=np.array(null_holl), null_ev1d=np.array(null_ev))

        if st == "wake":
            true_hd = nearest_hd(hd, t, BIN)
            ok = np.isfinite(true_hd)
            res["true_hd"] = true_hd
            res["decode_err"] = np.abs(circ_diff(dec[ok], true_hd[ok]))
            res["decode_circ_corr"] = circ_corr(dec[ok], true_hd[ok])
            hd_m = true_hd[mkeep][sel]
            m = np.isfinite(hd_m)
            res["true_hd_emb"] = hd_m
            res["map_conc_true"] = map_concentration(theta_emb[m], hd_m[m])

        # control ensemble: simultaneously recorded non-HD units, same pipeline
        if ctrl_spikes is not None:
            tc_ctrl = np.nan_to_num(tc[ctrl].values)
            pref_c = mvl_pref(tc_ctrl)[1]
            _, xc = population_matrix(ctrl_spikes, e, BIN, SMOOTH_SD)
            cc = np.asarray(ctrl_spikes.count(BIN, e).values, dtype=float)
            m = xc.sum(1) > 0
            xc, cc = xc[m], cc[m]
            dc, rc, _ = pairwise_corr_vs_angle(xc, pref_c)
            _, mxc = manifold_input(cc, MANIFOLD_SD, ACT_PCT)
            csel, cemb = ring_embedding(mxc, seed=seed)
            res["ctrl"] = dict(cos_r2=cosine_fit(dc, rc)[2],
                               hollowness=ring_stats(cemb)["hollowness"],
                               emb=cemb,
                               participation_ratio=participation_ratio(
                                   dimensionality_input(cc, SMOOTH_SD)),
                               pca_spectrum=pca_spectrum(
                                   dimensionality_input(cc, SMOOTH_SD), 10),
                               n=xc.shape[1])
        out[st] = res
        if verbose:
            print(f"  {st}: {x.shape[0]} bins  cos R2={r2:.2f} (null {np.mean(null_r2):.2f})  "
                  f"hollow={rs['hollowness']:.2f} (null {np.mean(null_holl):.2f})  "
                  f"cv-1D-EV={ev1d:.2f} (null {np.mean(null_ev):.2f})  "
                  f"map-conc={map_conc:.2f} (null {null_map:.2f})  "
                  f"split-half coh={coh:.2f} (null {np.mean(null_coh):.2f})", flush=True)
    return out


if False:
    for sid in sys.argv[1:]:
        r = analyze(sid)
        np.save(f"{sid}_results.npy", r, allow_pickle=True)

## Figures

In [ ]:
matplotlib.use("Agg")


TWO_PI = 2 * np.pi
STATE_COLORS = {"wake": "#1b6ca8", "REM": "#c0392b", "nREM": "#7d3c98"}
STATE_LABEL = {"wake": "Wake", "REM": "REM sleep", "nREM": "non-REM sleep"}
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.spines.top": False,
                     "axes.spines.right": False, "savefig.bbox": "tight"})


def _hd_raster(ax, spikes, units, pref, ep_window, t_origin=0.0):
    order = np.argsort(pref)
    cmap = plt.get_cmap("hsv")
    for row, i in enumerate(order):
        st = spikes[units[i]].get(ep_window[0], ep_window[1]).index.values - t_origin
        ax.plot(st, np.full_like(st, row), "|", ms=3, color=cmap(pref[i] / TWO_PI), mew=0.8)
    ax.set_ylim(-1, len(order))
    ax.set_xlim(ep_window[0] - t_origin, ep_window[1] - t_origin)


def smooth_decode(counts, tc, bin_size, win=3):
    """Poisson MAP decoding from counts pooled over `win` bins, for display."""
    from scipy.ndimage import uniform_filter1d
    c = uniform_filter1d(np.asarray(counts, float), win, axis=0) * win
    return bayesian_decode(c, tc, bin_size * win)[0]


# --------------------------------------------------------------- figure 1 ---
def _best_wake_window(hd, wake, dur=60.0):
    """Pick the window of `dur` seconds in which the head direction moves the most."""
    best, best_disp = None, -1
    for a, b in zip(wake.start, wake.end):
        if b - a < dur:
            continue
        for t0 in np.arange(a, b - dur, dur):
            h = hd.get(t0, t0 + dur).values
            if len(h) < 100:
                continue
            disp = 1 - np.abs(np.exp(1j * h).mean())
            if disp > best_disp:
                best_disp, best = disp, (t0, t0 + dur)
    return best


def fig_overview(res, s, fname):
    hd, spikes, ep = s["hd"], s["spikes"], s["epochs"]
    units, pref = res["hd_units"], res["pref"]
    fig = plt.figure(figsize=(12.5, 8.5))
    gs = GridSpec(3, 2, height_ratios=[0.55, 1.4, 1.4], hspace=0.55, wspace=0.25)

    # hypnogram
    ax = fig.add_subplot(gs[0, :])
    for lab, c, y in [("Awake", "#1b6ca8", 2), ("REM", "#c0392b", 1), ("Non-REM", "#7d3c98", 0)]:
        e = ep[lab]
        for a, b in zip(e.start, e.end):
            ax.add_patch(plt.Rectangle((a / 60, y - 0.35), (b - a) / 60, 0.7, color=c))
    ax.set_ylim(-0.6, 2.6); ax.set_yticks([0, 1, 2])
    ax.set_yticklabels(["non-REM", "REM", "awake"])
    ax.set_xlim(0, max(ep["Non-REM"].end.max(), ep["Awake"].end.max()) / 60)
    ax.set_xlabel("time in session (min)")
    ax.set_title(f"{res['sid']}  ({res['subject']}): brain-state segmentation, "
                 f"{res['n_units']} units, {len(units)} head-direction cells")

    # wake: raster over the most head-mobile minute, with the measured direction below
    t0, t1 = _best_wake_window(hd, ep["wake_tracked"])
    ax = fig.add_subplot(gs[1, 0])
    _hd_raster(ax, spikes, units, pref, (t0, t1), t0)
    ax.set_ylabel("HD cell (sorted by\npreferred direction)")
    ax.set_title("Wake: the bump follows the head")
    ax2 = fig.add_subplot(gs[2, 0], sharex=ax)
    h = hd.get(t0, t1)
    ax2.plot(h.index.values - t0, np.degrees(h.values), ".", ms=1.5, color="k")
    ax2.set_ylabel("measured head\ndirection (deg)"); ax2.set_xlabel("time (s)")
    ax2.set_ylim(0, 360); ax2.set_yticks([0, 180, 360])

    # sleep: raster with the decoded internal heading on top
    for row, (st, dur) in enumerate([("REM", 60.0), ("nREM", 25.0)]):
        r = res[st]
        t, dec = r["t"], smooth_decode(r["counts"], res["tc_hd"], res["bin"])
        eps = _episodes(t, res["bin"], min_bins=int(dur / res["bin"]))
        i0, i1 = max(eps, key=lambda e: e[1] - e[0])
        i1 = min(i1, i0 + int(dur / res["bin"]))
        ax = fig.add_subplot(gs[row + 1, 1])
        _hd_raster(ax, spikes, units, pref, (t[i0], t[i1 - 1]), t[i0])
        ax.plot(t[i0:i1] - t[i0], dec[i0:i1] / TWO_PI * len(units), ".", ms=3, color="k",
                label="internal heading decoded from these spikes")
        ax.set_title(f"{STATE_LABEL[st]}: same ensemble, animal asleep and still",
                     color=STATE_COLORS[st])
        ax.set_ylabel("HD cell (sorted)")
        ax.legend(fontsize=7, markerscale=2.5, loc="lower left",
                  bbox_to_anchor=(0, 1.0), frameon=False)
        if row == 1:
            ax.set_xlabel("time (s)")
    fig.savefig(fname)
    plt.close(fig)


# --------------------------------------------------------------- figure 2 ---
def fig_tuning(res, fname, n_show=24):
    tc = res["tc_hd"]; pref = res["pref"]; df = res["screen"]
    order = np.argsort(pref)
    n = min(n_show, tc.shape[1])
    ncol = 6
    nrow = int(np.ceil(n / ncol)) + 1
    fig = plt.figure(figsize=(13, 2.4 * nrow))
    gs = GridSpec(nrow, ncol, hspace=0.75, wspace=0.45)
    a = np.append(ANG, ANG[0])
    for k in range(n):
        i = order[int(round(k * (tc.shape[1] - 1) / max(n - 1, 1)))]
        ax = fig.add_subplot(gs[k // ncol, k % ncol], projection="polar")
        v = np.append(tc[:, i], tc[0, i])
        ax.plot(a, v, color=plt.get_cmap("hsv")(pref[i] / TWO_PI), lw=1.3)
        ax.fill(a, v, color=plt.get_cmap("hsv")(pref[i] / TWO_PI), alpha=0.25)
        ax.set_xticks(np.arange(0, TWO_PI, np.pi / 2))
        ax.set_xticklabels(["0", "", "180", ""], fontsize=6)
        ax.set_yticklabels([])
        ax.set_title(f"unit {res['hd_units'][i]}  ({v.max():.0f} Hz)", fontsize=7, pad=10)

    r = nrow - 1
    ax = fig.add_subplot(gs[r, 0:2], projection="polar")
    ax.hist(pref, bins=18, range=(0, TWO_PI), color="#444", alpha=0.8)
    ax.set_title("preferred directions\ntile the circle", fontsize=9, pad=26)
    ax.set_yticklabels([])

    ax = fig.add_subplot(gs[r, 2:4])
    ax.scatter(df["mvl_null99"], df["mvl"], s=16,
               c=["#c0392b" if h else "#bbb" for h in df["is_hd"]])
    lim = [0, max(np.nanmax(df["mvl"]), np.nanmax(df["mvl_null99"])) * 1.05]
    ax.plot(lim, lim, "k--", lw=0.8)
    ax.set_xlabel("99th pct of circular-shift null"); ax.set_ylabel("mean vector length")
    ax.set_title("HD selectivity vs shuffle null", fontsize=9)

    ax = fig.add_subplot(gs[r, 4:6])
    ax.scatter(df["stability"], df["mvl"], s=16,
               c=["#c0392b" if h else "#bbb" for h in df["is_hd"]])
    ax.axvline(0.5, ls="--", lw=0.8, c="k"); ax.axhline(0.25, ls="--", lw=0.8, c="k")
    ax.set_xlabel("split-half tuning-curve correlation"); ax.set_ylabel("mean vector length")
    ax.set_title("tuning stability within wake", fontsize=9)
    fig.suptitle(f"{res['sid']}: head-direction tuning during wake "
                 f"({len(pref)} HD cells of {res['n_units']} units)", y=1.0)
    fig.savefig(fname)
    plt.close(fig)


# --------------------------------------------------------------- figure 3 ---
def fig_correlations(res, fname):
    fig, axes = plt.subplots(2, 3, figsize=(13, 7.5))
    order = res["order"]
    for j, st in enumerate(["wake", "REM", "nREM"]):
        r = res[st]
        c = r["cmat"][np.ix_(order, order)].copy()
        np.fill_diagonal(c, np.nan)
        v = np.nanpercentile(np.abs(c), 98)
        ax = axes[0, j]
        im = ax.imshow(c, cmap="RdBu_r", vmin=-v, vmax=v)
        ax.set_title(f"{STATE_LABEL[st]}\npairwise correlations", color=STATE_COLORS[st])
        ax.set_xlabel("HD cell (sorted by preferred direction)")
        if j == 0:
            ax.set_ylabel("HD cell (sorted)")
        plt.colorbar(im, ax=ax, fraction=0.046, label="r")

        ax = axes[1, j]
        d, rr = r["corr_d"], r["corr_r"]
        ax.scatter(np.degrees(d), rr, s=8, alpha=0.35, color=STATE_COLORS[st])
        b = np.linspace(0, np.pi, 13)
        idx = np.digitize(d, b) - 1
        m = np.array([np.mean(rr[idx == k]) if (idx == k).sum() else np.nan
                      for k in range(len(b) - 1)])
        bc = np.degrees((b[:-1] + b[1:]) / 2)
        ax.plot(bc, m, "o-", color="k", ms=4, lw=1.5, label="binned mean")
        xx = np.linspace(0, np.pi, 200)
        ax.plot(np.degrees(xx), r["cos_a"] + r["cos_b"] * np.cos(xx), "--", color="k", lw=1.2,
                label=f"cosine fit, $R^2$={r['cos_r2']:.2f}")
        ax.axhline(0, color="gray", lw=0.6)
        ax.set_xlabel("|Δ preferred direction| (deg, from wake)")
        if j == 0:
            ax.set_ylabel("correlation of binned firing (100 ms)")
        ax.set_title(f"shuffle null $R^2$ = {r['null_cos_r2'].mean():.3f}", fontsize=9)
        ax.legend(fontsize=7, loc="upper right")
        ax.set_xticks([0, 45, 90, 135, 180])
    fig.suptitle(f"{res['sid']}: the wake correlation structure of the HD ensemble "
                 "is preserved in both sleep states", y=0.98)
    fig.tight_layout()
    fig.savefig(fname)
    plt.close(fig)


# --------------------------------------------------------------- figure 4 ---
def fig_manifold(res, fname):
    fig = plt.figure(figsize=(13, 8))
    gs = GridSpec(2, 3, hspace=0.35, wspace=0.35)
    for j, st in enumerate(["wake", "REM", "nREM"]):
        r = res[st]
        emb = r["emb"]
        col = (r["true_hd_emb"] if st == "wake"
               else smooth_decode(r["counts"], res["tc_hd"], res["bin"])[r["emb_keep"]][r["emb_sel"]])
        ax = fig.add_subplot(gs[0, j])
        sc = ax.scatter(emb[:, 0], emb[:, 1], c=col, cmap="hsv", s=5, vmin=0, vmax=TWO_PI)
        ax.set_title(f"{STATE_LABEL[st]}\nhollowness = {r['hollowness']:.2f} "
                     f"(null {r['null_hollowness'].mean():.2f})", color=STATE_COLORS[st])
        ax.set_xlabel("Isomap 1"); ax.set_ylabel("Isomap 2" if j == 0 else "")
        ax.set_aspect("equal")
        cb = plt.colorbar(sc, ax=ax, fraction=0.046, ticks=[0, np.pi, TWO_PI])
        cb.ax.set_yticklabels(["0", "180", "360"])
        cb.set_label("measured HD (deg)" if st == "wake"
                     else "decoded internal HD (deg, 300 ms)", fontsize=8)

    r = res["nREM"]
    ax = fig.add_subplot(gs[1, 0])
    _, mxs = manifold_input(shift_shuffle(r["counts"], np.random.default_rng(1)))
    _, semb = ring_embedding(mxs, max_points=2500, seed=1)
    ax.scatter(semb[:, 0], semb[:, 1], s=5, color="#999")
    ax.set_aspect("equal"); ax.set_xlabel("Isomap 1"); ax.set_ylabel("Isomap 2")
    ax.set_title(f"non-REM, time-shifted control\nhollowness = "
                 f"{ring_stats(semb)['hollowness']:.2f}")

    ax = fig.add_subplot(gs[1, 1])
    if "ctrl" in r:
        ax.scatter(r["ctrl"]["emb"][:, 0], r["ctrl"]["emb"][:, 1], s=5, color="#999")
        ax.set_title(f"non-REM, non-HD units (n={r['ctrl']['n']})\n"
                     f"hollowness = {r['ctrl']['hollowness']:.2f}")
    ax.set_aspect("equal"); ax.set_xlabel("Isomap 1"); ax.set_ylabel("Isomap 2")

    ax = fig.add_subplot(gs[1, 2])
    for st in ["wake", "REM", "nREM"]:
        e = res[st]["emb"]
        rad = np.hypot(*(e - taubin_center(e)).T)
        ax.hist(rad / rad.mean(), bins=60, range=(0, 2.2), histtype="step", density=True,
                color=STATE_COLORS[st], label=STATE_LABEL[st], lw=1.4)
    rad = np.hypot(*(semb - taubin_center(semb)).T)
    ax.hist(rad / rad.mean(), bins=60, range=(0, 2.2), histtype="step", density=True,
            color="#999", label="non-REM, time-shifted", lw=1.4, ls="--")
    ax.set_xlabel("radius in embedding / mean radius")
    ax.set_ylabel("density")
    ax.set_title("a ring is hollow, a blob is not")
    ax.legend(fontsize=7)
    fig.suptitle(f"{res['sid']}: population states lie on a closed one-dimensional ring "
                 "in every brain state", y=0.98)
    fig.savefig(fname)
    plt.close(fig)


# --------------------------------------------------------------- figure 5 ---
def fig_dimensionality(res, fname):
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
    ax = axes[0]
    for st in ["wake", "REM", "nREM"]:
        sp = res[st]["pca_spectrum"]
        ax.plot(np.arange(1, len(sp) + 1), sp * 100, "o-", ms=4,
                color=STATE_COLORS[st], label=STATE_LABEL[st])
    ctrl_spec = res["nREM"].get("ctrl", {}).get("pca_spectrum")
    if ctrl_spec is not None:
        ax.plot(np.arange(1, len(ctrl_spec) + 1), ctrl_spec * 100, "o--", ms=4,
                color="#999", label="non-HD units (nREM)")
    ax.set_xlabel("principal component"); ax.set_ylabel("variance explained (%)")
    ax.set_title("PCA spectrum (200 ms smoothed)"); ax.legend(fontsize=7)

    ax = axes[1]
    vals = [res[st]["pca_spectrum"][1] / res[st]["pca_spectrum"][0]
            for st in ["wake", "REM", "nREM"]]
    ax.bar(np.arange(3), vals, color=[STATE_COLORS[s] for s in ["wake", "REM", "nREM"]])
    if ctrl_spec is not None:
        ax.bar(3, ctrl_spec[1] / ctrl_spec[0], color="#999")
    ax.axhline(1.0, ls="--", lw=0.8, color="k")
    ax.set_xticks(np.arange(4)); ax.set_xticklabels(["wake", "REM", "nREM", "non-HD\n(nREM)"])
    ax.set_ylabel("PC2 variance / PC1 variance")
    ax.set_title("a ring spans a plane, so its two\nleading components are equal")

    ax = axes[2]
    for i, st in enumerate(["wake", "REM", "nREM"]):
        r = res[st]
        ax.bar(i - 0.18, r["hollowness"], 0.34, color=STATE_COLORS[st])
        ax.bar(i + 0.18, r["null_hollowness"].mean(), 0.34, color="#ccc",
               yerr=r["null_hollowness"].std(), error_kw=dict(lw=0.8))
    ax.set_xticks(range(3)); ax.set_xticklabels(["wake", "REM", "nREM"])
    ax.set_ylabel("hollowness  (mean r / SD r)")
    ax.set_title("ring topology vs time-shift null\n(grey = null)")

    ax = axes[3]
    for i, st in enumerate(["wake", "REM", "nREM"]):
        r = res[st]
        ax.bar(i - 0.18, r["ev1d"], 0.34, color=STATE_COLORS[st])
        ax.bar(i + 0.18, r["null_ev1d"].mean(), 0.34, color="#ccc",
               yerr=r["null_ev1d"].std(), error_kw=dict(lw=0.8))
    ax.set_xticks(range(3)); ax.set_xticklabels(["wake", "REM", "nREM"])
    ax.set_ylabel("cross-validated mean $R^2$")
    ax.set_title("held-out neurons explained by the\nangle decoded from the others")
    fig.suptitle(f"{res['sid']}: the variance of the HD ensemble concentrates in an "
                 "equal-variance plane that holds the ring", y=1.04)
    fig.tight_layout()
    fig.savefig(fname)
    plt.close(fig)


# --------------------------------------------------------------- figure 6 ---
def fig_coherence(res, fname, seed=0):
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
    ax = axes[0]
    for i, st in enumerate(["wake", "REM", "nREM"]):
        r = res[st]
        ax.bar(i - 0.18, r["split_coh"], 0.34, color=STATE_COLORS[st])
        ax.bar(i + 0.18, r["null_coh"].mean(), 0.34, color="#ccc",
               yerr=r["null_coh"].std(), error_kw=dict(lw=0.8))
    ax.set_xticks(range(3)); ax.set_xticklabels(["wake", "REM", "nREM"])
    ax.set_ylabel("resultant length of the angular difference")
    ax.set_ylim(0, 1)
    ax.set_title("split-ensemble decoder agreement\n(grey = time-shift null)")

    for j, st in enumerate(["REM", "nREM"]):
        r = res[st]
        da, db = split_half_decode(r["counts"], res["tc_hd"], res["bin"],
                                   np.random.default_rng(seed))
        ax = axes[1 + j]
        ax.hist2d(np.degrees(da), np.degrees(db), bins=48, range=[[0, 360], [0, 360]],
                  cmap="magma", norm="log")
        ax.set_xlabel("heading decoded from ensemble half A (deg)")
        ax.set_ylabel("half B (deg)")
        ax.set_title(f"{STATE_LABEL[st]}\nagreement = {r['split_coh']:.2f}",
                     color=STATE_COLORS[st])
        ax.set_xticks([0, 180, 360]); ax.set_yticks([0, 180, 360])

    ax = axes[3]
    k = int(round(1.0 / res["bin"]))
    for st in ["wake", "REM", "nREM"]:
        r = res[st]
        d = smooth_decode(r["counts"], res["tc_hd"], res["bin"])
        t = r["t"]
        ok = (t[k:] - t[:-k]) < 1.5
        v = np.degrees(np.abs(circ_diff(d[k:], d[:-k]))[ok])
        ax.hist(v, bins=np.linspace(0, 180, 60), histtype="step", density=True,
                color=STATE_COLORS[st], lw=1.4,
                label=f"{STATE_LABEL[st]} (median {np.median(v):.0f}°/s)")
    ax.set_xlim(0, 180)
    ax.set_xlabel("|angular displacement| of the internal heading over 1 s (deg)")
    ax.set_ylabel("density"); ax.legend(fontsize=7)
    ax.set_title("the internal heading moves smoothly,\nfaster in non-REM than in REM")
    fig.suptitle(f"{res['sid']}: the ring coordinate is internally coherent during sleep",
                 y=1.04)
    fig.tight_layout()
    fig.savefig(fname)
    plt.close(fig)


# --------------------------------------------------------------- figure 7 ---
def fig_decode_validation(res, fname):
    r = res["wake"]
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
    ax = axes[0]
    t, dec, true = r["t"], r["decoded"], r["true_hd"]
    # show the 90 s stretch in which the animal actually turned its head the most
    w = 900
    best, best_disp = (0, w), -1.0
    for a, b in _episodes(t, res["bin"], min_bins=w):
        for i0 in range(a, b - w, w):
            h = true[i0:i0 + w]; h = h[np.isfinite(h)]
            if len(h) < w // 2:
                continue
            disp = 1 - np.abs(np.exp(1j * h).mean())
            if disp > best_disp:
                best_disp, best = disp, (i0, i0 + w)
    i0, i1 = best
    ax.plot(t[i0:i1] - t[i0], np.degrees(true[i0:i1]), ".", ms=2, color="k", label="measured HD")
    ax.plot(t[i0:i1] - t[i0], np.degrees(dec[i0:i1]), ".", ms=2, color="#1b6ca8",
            label="decoded from HD cells")
    ax.set_xlabel("time (s)"); ax.set_ylabel("head direction (deg)")
    ax.set_yticks([0, 180, 360]); ax.legend(fontsize=7, markerscale=4)
    ax.set_title("wake: decoder validation")

    ax = axes[1]
    ax.hist(np.degrees(r["decode_err"]), bins=60, color="#1b6ca8")
    ax.axvline(np.degrees(np.median(r["decode_err"])), color="k", ls="--")
    ax.set_xlabel("|decoding error| (deg)"); ax.set_ylabel("bins")
    ax.set_title(f"median error {np.degrees(np.median(r['decode_err'])):.0f}°, "
                 f"circ. r = {r['decode_circ_corr']:.2f}")

    ax = axes[2]
    for i, st in enumerate(["wake", "REM", "nREM"]):
        ax.bar(i - 0.18, res[st]["map_conc"], 0.34, color=STATE_COLORS[st])
        ax.bar(i + 0.18, res[st]["null_map_conc"], 0.34, color="#ccc")
    ax.set_ylabel("resultant length of the residual")
    ax.set_xticks(range(3)); ax.set_xticklabels(["wake", "REM", "nREM"])
    ax.set_ylim(0, 1)
    ax.set_title("how sharply the ring angle fixes the\ndecoded heading (grey = shuffle)")
    fig.suptitle(f"{res['sid']}: the ring coordinate recovered without any behavioural "
                 "reference matches the wake heading code", y=1.04)
    fig.tight_layout()
    fig.savefig(fname)
    plt.close(fig)


# --------------------------------------------------------------- figure 8 ---
def fig_multisession(summary, fname):
    import pandas as pd
    df = pd.DataFrame(summary)
    metrics = [("cos_r2", "cosine fit $R^2$\n(corr. vs Δpreferred dir.)"),
               ("hollowness", "ring hollowness\n(mean r / SD r)"),
               ("ev1d", "held-out neurons explained by\none angular coordinate ($R^2$)"),
               ("split_coh", "split-ensemble decoder\nagreement (resultant)"),
               ("map_conc", "ring angle fixes the\ndecoded heading (resultant)")]
    fig, axes = plt.subplots(1, 5, figsize=(17.5, 3.8))
    states = ["wake", "REM", "nREM"]
    for ax, (m, lab) in zip(axes, metrics):
        for i, st in enumerate(states):
            v = df[f"{m}_{st}"].values
            nv = df[f"null_{m}_{st}"].values
            ax.plot(np.full_like(v, i - 0.16) + np.random.uniform(-.04, .04, len(v)), v,
                    "o", ms=5, color=STATE_COLORS[st], alpha=0.85)
            ax.plot(np.full_like(nv, i + 0.16) + np.random.uniform(-.04, .04, len(nv)), nv,
                    "o", ms=5, color="#bbb")
            ax.hlines(np.mean(v), i - 0.30, i - 0.02, color=STATE_COLORS[st], lw=2)
            ax.hlines(np.mean(nv), i + 0.02, i + 0.30, color="#888", lw=2)
        ax.set_xticks(range(3)); ax.set_xticklabels(states)
        ax.set_title(lab, fontsize=9)
    axes[0].set_ylabel("value  (coloured = data, grey = null)")
    fig.suptitle(f"Across {len(df)} sessions from {df.subject.nunique()} mice: "
                 "the one-dimensional ring is present in wake, REM and non-REM", y=1.03)
    fig.tight_layout()
    fig.savefig(fname)
    plt.close(fig)
    return df


# --------------------------------------------------------------- figure 9 ---
def _episodes(t, bin_size, min_bins=300):
    b = np.concatenate([[0], np.where(np.diff(t) > 1.5 * bin_size)[0] + 1, [len(t)]])
    return [(b[i], b[i + 1]) for i in range(len(b) - 1) if b[i + 1] - b[i] >= min_bins]


def fig_sleep_vs_behavior(res, s, fname, nearest_hd=None):
    """The decoded internal heading travels the ring while the animal's head does not."""
    hd = s["hd"]
    bs = res["bin"]
    fig, axes = plt.subplots(2, 2, figsize=(13, 7.5))
    meas = {st: nearest_hd(hd, res[st]["t"], bs) for st in ["wake", "REM", "nREM"]}
    stats = {}

    for row, st in enumerate(["REM", "nREM"]):
        r = res[st]
        t, dec, m = r["t"], r["decoded"], meas[st]
        eps = _episodes(t, bs)
        i0, i1 = max(eps, key=lambda e: e[1] - e[0])
        i1 = min(i1, i0 + 1200)
        ax = axes[row, 0]
        ax.plot(t[i0:i1] - t[i0], np.degrees(m[i0:i1]), ".", ms=2, color="#999",
                label="measured head direction")
        ax.plot(t[i0:i1] - t[i0], np.degrees(dec[i0:i1]), ".", ms=2,
                color=STATE_COLORS[st], label="decoded internal heading")
        ax.set_ylim(0, 360); ax.set_yticks([0, 180, 360])
        ax.set_ylabel("direction (deg)")
        ax.set_title(f"{STATE_LABEL[st]}: the head is still, the internal heading is not",
                     color=STATE_COLORS[st])
        ax.legend(fontsize=7, markerscale=4, loc="upper right")
        if row == 1:
            ax.set_xlabel("time within sleep episode (s)")

    # per-episode angular concentration of the two signals
    ax = axes[0, 1]
    for i, st in enumerate(["wake", "REM", "nREM"]):
        r = res[st]
        Rm, Rd = [], []
        for a, b in _episodes(r["t"], bs):
            mm = meas[st][a:b]; mm = mm[np.isfinite(mm)]
            if len(mm) < 100:
                continue
            Rm.append(np.abs(np.exp(1j * mm).mean()))
            Rd.append(np.abs(np.exp(1j * r["decoded"][a:b]).mean()))
        stats[f"R_head_{st}"] = float(np.median(Rm))
        stats[f"R_decoded_{st}"] = float(np.median(Rd))
        stats[f"n_episodes_{st}"] = len(Rm)
        jit = np.random.uniform(-0.06, 0.06, len(Rm))
        ax.plot(i - 0.18 + jit, Rm, "o", ms=4, color="#999", alpha=0.6)
        ax.plot(i + 0.18 + jit, Rd, "o", ms=4, color=STATE_COLORS[st], alpha=0.6)
        ax.hlines(np.median(Rm), i - 0.33, i - 0.03, color="k", lw=2)
        ax.hlines(np.median(Rd), i + 0.03, i + 0.33, color="k", lw=2)
    ax.set_xticks(range(3)); ax.set_xticklabels(["wake", "REM", "nREM"])
    ax.set_ylabel("concentration within episode (resultant length)")
    ax.set_ylim(0, 1.05)
    ax.set_title("grey: measured head direction   colour: decoded internal heading\n"
                 "in sleep the head is fixed while the internal heading sweeps the ring",
                 fontsize=9)

    # median angular speed over a 1 s window, head vs internal
    ax = axes[1, 1]
    w = 1.0; k = int(round(w / bs))
    for i, st in enumerate(["wake", "REM", "nREM"]):
        r = res[st]
        t, dec, m = r["t"], r["decoded"], meas[st]
        ok = ((t[k:] - t[:-k]) < 1.5 * w) & np.isfinite(m[k:]) & np.isfinite(m[:-k])
        vh = np.degrees(np.median(np.abs(circ_diff(m[k:], m[:-k]))[ok] / w))
        vd = np.degrees(np.median(np.abs(circ_diff(dec[k:], dec[:-k]))[ok] / w))
        stats[f"head_speed_{st}"] = float(vh)
        stats[f"internal_speed_{st}"] = float(vd)
        ax.bar(i - 0.18, vh, 0.34, color="#999")
        ax.bar(i + 0.18, vd, 0.34, color=STATE_COLORS[st])
    ax.set_xticks(range(3)); ax.set_xticklabels(["wake", "REM", "nREM"])
    ax.set_ylabel("median |angular speed| over 1 s (deg/s)")
    ax.set_title("grey: real head movement   colour: internal heading", fontsize=9)

    fig.suptitle(f"{res['sid']}: during sleep the ring state is driven internally, "
                 "not by the animal's head", y=1.0)
    fig.tight_layout()
    fig.savefig(fname)
    plt.close(fig)
    return stats

## Run

The primary session is `Mouse28-140312` (25 HD cells of
72 units). Results for each session are cached to
`cache_<session>.npy`, so re-running the notebook does not repeat the analysis.

In [ ]:
PRIMARY = 'Mouse28-140312'
ALL_SESSIONS = ['Mouse12-120807', 'Mouse25-140130', 'Mouse28-140312', 'Mouse28-140313']

def get(sid):
    fn = f'cache_{sid}.npy'
    if os.path.exists(fn):
        return np.load(fn, allow_pickle=True).item()
    r = analyze(sid)
    np.save(fn, r, allow_pickle=True)
    return r

res = {sid: get(sid) for sid in ALL_SESSIONS}
r = res[PRIMARY]
s = load_session(SESSIONS[PRIMARY])

### Raw data

Brain-state segmentation, the wake spike raster of the HD ensemble sorted by preferred
direction next to the measured head direction, and the same raster during REM and
non-REM sleep. The diagonal streaks in the sleep rasters are the activity bump moving
around the ring; the black dots are the heading decoded from those same spikes.

In [ ]:
fig_overview(r, s, 'fig01_data_overview.png')

### Wake tuning curves and cell selection

In [ ]:
fig_tuning(r, 'fig02_hd_tuning_curves.png')

### Correlation structure

The correlation matrix sorted by preferred direction has the same banded, wrap-around
structure in all three states. The cosine fit explains
R² = 0.51 of the pair-correlation variance in wake,
0.48 in REM and 0.41 in non-REM, against
0.004 for the time-shifted null in non-REM.

In [ ]:
fig_correlations(r, 'fig03_pairwise_correlations.png')

### The manifold is a ring in every state

Hollowness is 4.46 in wake, 5.94 in REM and
3.45 in non-REM, versus 2.15 for the
time-shifted null and 2.65 for a same-size ensemble of non-HD
units recorded simultaneously. Colour is the measured head direction in wake and the
decoded internal heading in sleep; in both cases it advances monotonically around the
ring, which is what makes this a *continuous* attractor rather than a set of discrete
states.

In [ ]:
fig_manifold(r, 'fig04_ring_manifold.png')

### Dimensionality

The two leading principal components of the HD ensemble carry nearly equal variance
(PC2/PC1 = 0.82 in non-REM and 0.93 in wake, against
0.50 for the non-HD control), which is what a ring in a plane
looks like. The participation ratio itself stays high (14.09 in non-REM)
because 100 ms spike counts are noise-dominated; that is why the ring is quantified
with the manifold and decoding measures instead. A single
angle decoded from one half of the ensemble accounts for R² = 0.13 of the
variance of the *other*, held-out half in non-REM (0.19 in wake), against
0.00 for the null.

In [ ]:
fig_dimensionality(r, 'fig05_dimensionality.png')

### Internal coherence and continuity of movement

Two disjoint halves of the ensemble, decoding independently, agree with resultant
length 0.67 in REM and 0.62 in non-REM
(0.71 in wake), against 0.10 for the null. The
internal heading also moves at a finite speed rather than jumping, and it moves faster
in non-REM than in REM.

In [ ]:
fig_coherence(r, 'fig06_internal_coherence.png')

### The ring coordinate is the heading coordinate

During wake the decoder recovers the animal's true head direction with a median error
of 12° (circular r = 0.84), which
validates both the tuning curves and the decoder. The angular position on the Isomap
ring, computed with no behavioural or tuning-curve information at all, pins down the
decoded heading with residual resultant length 0.71 in non-REM and
0.94 in REM, against 0.20 for shuffled labels.

In [ ]:
fig_decode_validation(r, 'fig07_decoding_validation.png')

### The sleeping animal's head is not driving the ring

The strongest form of the internal-maintenance claim is a direct comparison. Within
sleep episodes of at least 30 s the measured head direction is essentially fixed
(median resultant length 1.00 in REM and 1.00 in non-REM)
while the decoded internal heading is spread around the ring (0.35 and
0.37). Over a 1 s window the head turns at
2°/s in REM and 3°/s in non-REM
while the internal heading travels 23°/s and
68°/s.

In [ ]:
beh = fig_sleep_vs_behavior(r, s, 'fig09_internal_vs_measured.png',
                            nearest_hd=nearest_hd)

### Across sessions

In [ ]:
def summarize(r):
    row = dict(sid=r["sid"], subject=r["subject"], n_units=r["n_units"],
               n_hd=len(r["hd_units"]))
    for st in ["wake", "REM", "nREM"]:
        d = r[st]
        row[f"cos_r2_{st}"] = d["cos_r2"]
        row[f"null_cos_r2_{st}"] = d["null_cos_r2"].mean()
        row[f"hollowness_{st}"] = d["hollowness"]
        row[f"null_hollowness_{st}"] = d["null_hollowness"].mean()
        row[f"ev1d_{st}"] = float(d["ev1d"])
        row[f"null_ev1d_{st}"] = d["null_ev1d"].mean()
        row[f"split_coh_{st}"] = d["split_coh"]
        row[f"null_split_coh_{st}"] = d["null_coh"].mean()
        row[f"map_conc_{st}"] = d["map_conc"]
        row[f"null_map_conc_{st}"] = d["null_map_conc"]
        row[f"drift_median_{st}"] = float(np.degrees(np.median(d["drift"])))
        row[f"pr_{st}"] = d["participation_ratio"]
        row[f"pc2_pc1_{st}"] = float(d["pca_spectrum"][1] / d["pca_spectrum"][0])
        row[f"dur_{st}"] = r["ep_len"][st]
    row["decode_err_median_deg"] = float(np.degrees(np.median(r["wake"]["decode_err"])))
    row["decode_circ_corr"] = r["wake"]["decode_circ_corr"]
    if "ctrl" in r["nREM"]:
        row["ctrl_hollowness_nREM"] = r["nREM"]["ctrl"]["hollowness"]
        row["ctrl_cos_r2_nREM"] = r["nREM"]["ctrl"]["cos_r2"]
        cs = r["nREM"]["ctrl"].get("pca_spectrum")
        if cs is not None:
            row["ctrl_pc2_pc1_nREM"] = float(cs[1] / cs[0])
        row["ctrl_n"] = r["nREM"]["ctrl"]["n"]
    return row

In [ ]:
summary = [summarize(res[sid]) for sid in ALL_SESSIONS]
summary[ALL_SESSIONS.index(PRIMARY)].update(beh)
df = fig_multisession(summary, 'fig08_multisession_summary.png')
df.to_csv('session_summary.csv', index=False)
df[['sid', 'subject', 'n_hd', 'cos_r2_nREM', 'hollowness_nREM',
    'ev1d_nREM', 'split_coh_nREM']]

## Conclusion

Across 4 sessions from 3 mice, the head-direction ensemble keeps the same
one-dimensional ring geometry in REM and non-REM sleep that it has during waking
exploration. The evidence is that (i) pairwise correlations remain a cosine function of
the wake preferred-direction difference, (ii) the population state cloud is a hollow
ring whose angular coordinate is the heading coordinate, (iii) two disjoint halves of
the ensemble report the same angle, and (iv) that angle moves continuously. None of
these hold for time-shifted surrogates or for simultaneously recorded non-HD units,
so the structure is a property of the coordinated population and not of single-cell
statistics.

Because the sleep measurements use no behavioural signal and the animal is not
generating informative vestibular or visual heading input, the ring has to be
maintained by the network itself. That is the defining property of a continuous
attractor. The main quantitative difference between states is the speed at which the
bump travels: it drifts several times faster in non-REM than in REM, consistent with
the time-compressed replay-like dynamics reported for this system.

### Limitations

The units table in this NWB conversion has no anatomical labels, so anterior thalamic
and post-subicular cells are pooled and the analysis cannot separate them. Isomap
hollowness is a summary statistic, not a topological proof; a persistent-homology test
would be the stronger version of the same claim. Finally, sleep states are taken as
given from the archived scoring rather than re-derived from the LFP.